In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql://{os.getenv('DB_USERNAME')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Ready")

Ready


In [3]:
df = pd.read_csv("../datasets/clinic_attendance006.csv")
print(df.shape)
print(df.head())
print(df.columns.to_list())
print(df.info())

# Missing values
print(f"Missing Values {df.isnull().sum()}")

# Descriptive statistics
print(df.describe())

# Categorical columns
for col in ['gender', 'diagnosis', 'payment_status']:
    print(f"\n{col}:")
    print(df[col].value_counts())

(25, 17)
   visit_id    patient_name  age  gender  province  district  visit_date  \
0         1      Ahmad Shah   34    Male     Kabul     Kabul  2024-03-01   
1         2    Fatima Noori   28  female     Kabul     Kabul  2024-03-01   
2         3    Mohammad Ali   45    Male  Kandahar  Kandahar  2024-03-02   
3         4  Zainab Hussain   32  Female     Herat     Herat  2024-03-02   
4         5     Abdul Karim   67    MALE     Kabul     Kabul  2024-03-03   

  clinic_name doctor_name        diagnosis treatment_provided  \
0    Clinic_A   Dr_Ahmadi      respiratory         medication   
1    Clinic_A   Dr_Ahmadi  maternal_health            checkup   
2    Clinic_B   Dr_Karimi     malnutrition  nutrition_support   
3    Clinic_C  Dr_Sultani  maternal_health            checkup   
4    Clinic_A   Dr_Ahmadi      respiratory         medication   

  follow_up_required  visit_duration_minutes  patient_satisfaction  \
0                yes                      45                     4   
1  

### Data Cleaning

In [4]:
df_clean = df.copy()

# fix visit_date type
df_clean['visit_date'] = pd.to_datetime(df_clean['visit_date'])
print(df_clean['visit_date'].dtype)

# Standardize gender column to consistent cases
df_clean['gender'] = df_clean['gender'].str.capitalize()
print ( df_clean['gender'].value_counts())

# fix impossible value
df_clean['visit_duration_minutes'] = df_clean['visit_duration_minutes'].replace(999, np.nan)
print((df_clean['visit_duration_minutes'] == 999).sum())
print(df_clean['visit_duration_minutes'].max())
print(df_clean['visit_duration_minutes'].value_counts())
print(df_clean['visit_duration_minutes'].isnull().sum())


datetime64[us]
gender
Female    13
Male      12
Name: count, dtype: int64
0
70.0
visit_duration_minutes
30.0    5
40.0    5
45.0    3
60.0    2
50.0    2
25.0    2
35.0    2
55.0    1
65.0    1
70.0    1
Name: count, dtype: int64
1


In [5]:
print(df_clean.shape)

(25, 17)


In [6]:
print("=== Final Clean Check ===")
print(df_clean.isnull().sum())
print("\nData types:")
print(df_clean.dtypes)
print("\nShape:", df_clean.shape)
print("\nVisit duration stats after fix:")
print(df_clean['visit_duration_minutes'].describe())

=== Final Clean Check ===
visit_id                  0
patient_name              0
age                       0
gender                    0
province                  0
district                  0
visit_date                0
clinic_name               0
doctor_name               0
diagnosis                 0
treatment_provided        0
follow_up_required        0
visit_duration_minutes    1
patient_satisfaction      0
travel_distance_km        0
payment_status            0
amount_paid_usd           0
dtype: int64

Data types:
visit_id                           int64
patient_name                         str
age                                int64
gender                               str
province                             str
district                             str
visit_date                datetime64[us]
clinic_name                          str
doctor_name                          str
diagnosis                            str
treatment_provided                   str
follow_up_required   

## Data Quality Assessment - Clinic Attendance Dataset

**Dataset:** 25 rows, 17 columns

### Issues Found and Fixed

**1. Data Type = visit_date**
- Problem: visit_date was stored as string (object) instead of datetime
- Fix: Converted to datetime64 using pd.to_datetime()
- Why it matters: Date columns must be datetime for time-based analysis and filtering

**2. Inconsistent Gender Values**
- Problem: Gender had 4 variations = Male, male, MALE, Female
- Fix: Standardized using str.capitalize() → Male, Female only
- Why it matters: Inconsistent values split gender groups in analysis, 
  producing incorrect counts

**3. Impossible Value = visit_duration_minutes**
- Problem: One record had visit_duration_minutes = 999, 
  which is not a realistic clinic visit duration
- Fix: Replaced with NaN = left as missing rather than imputed
- Why it matters: 999 was distorting the mean from ~40 to 80+ minutes. 
  Filling with an estimated value would invent data we don't have. 
  NaN is the honest choice.

### No Issues Found
- No missing values across any column
- No duplicate records
- diagnosis, payment_status, province all consistent
- patient_satisfaction within valid range (1-5)